# [INFO] Glu-Stock: 02_SIGNAL_INFERENCE
**Phase**: Ensemble Intelligence (LightGBM + CNN) | v18.2 (Recursive Discovery)

This notebook performs dual-brain inference. It uses Recursive Search to find model files in nested Kaggle paths.

In [ ]:
# [INSTALL] SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib tensorflow python-dotenv ta lightgbm


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Universe)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite
from firebase_admin import credentials, firestore
from datetime import datetime
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw = user_secrets.get_secret("FIREBASE_KEY_JSON")
                return {"key": json.loads(raw)}
            except Exception as e:
                print(f"[ERROR] FIREBASE_KEY_JSON missing or invalid! Error: {e}")
                return {"key": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            raw = os.getenv("FIREBASE_KEY_JSON")
            if not raw: return {"key": None}
            return {"key": json.loads(raw)}

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing. Check Kaggle Secrets.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()

    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({
            'timestamp': datetime.now().isoformat(), 
            'phase': phase.upper(), 
            'details': details
        })

    def wait_for_queue(self, queue_name: str, max_retries=20, interval=60):
        import time
        for i in range(max_retries):
            docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
            if docs:
                return self.get_and_clear_queue(queue_name)
            if i < max_retries - 1:
                print(f"[WAIT] {queue_name} queue empty. Retrying ({i+1}/{max_retries}) in {interval}s...", flush=True)
                time.sleep(interval)
        return []

def get_full_idx_universe():
    fallback = ['AALI.JK', 'ABMM.JK', 'ACES.JK', 'ADHI.JK', 'AISA.JK', 'AKRA.JK', 'AMRT.JK', 'ANTM.JK', 'APLN.JK', 'ARNA.JK', 'ARTO.JK', 'ASGR.JK', 'ASII.JK', 'ASRI.JK', 'ASSA.JK', 'AUTO.JK', 'BACA.JK', 'BALI.JK', 'BAYU.JK', 'BBCA.JK', 'BBHI.JK', 'BBNI.JK', 'BBRI.JK', 'BBTN.JK', 'BBYB.JK', 'BCAP.JK', 'BDMN.JK', 'BEST.JK', 'BFIN.JK', 'BGTG.JK', 'BINA.JK', 'BIRD.JK', 'BISI.JK', 'BJBR.JK', 'BJTM.JK', 'BKSL.JK', 'BMRI.JK', 'BMTR.JK', 'BNGA.JK', 'BNII.JK', 'BNLI.JK', 'BRMS.JK', 'BRPT.JK', 'BSDE.JK', 'BSIM.JK', 'BTPN.JK', 'BUDI.JK', 'BUKK.JK', 'BUMI.JK', 'BVIC.JK', 'BWPT.JK', 'BYAN.JK', 'CASS.JK', 'CFIN.JK', 'CITA.JK', 'CMNP.JK', 'CPIN.JK', 'CTRA.JK', 'DEWA.JK', 'DILD.JK', 'DLTA.JK', 'DMAS.JK', 'DNET.JK', 'DOID.JK', 'DSNG.JK', 'DSSA.JK', 'ELSA.JK', 'EMTK.JK', 'ENRG.JK', 'ERAA.JK', 'ESSA.JK', 'EXCL.JK', 'GEMS.JK', 'GGRM.JK', 'GJTL.JK', 'GWSA.JK', 'HEXA.JK', 'HMSP.JK', 'HRUM.JK', 'ICBP.JK', 'IMAS.JK', 'IMPC.JK', 'INCO.JK', 'INDF.JK', 'INDY.JK', 'INKP.JK', 'INPC.JK', 'INTP.JK', 'ISAT.JK', 'ISSP.JK', 'ITMG.JK', 'JKON.JK', 'JPFA.JK', 'JRPT.JK', 'JSMR.JK', 'JTPE.JK', 'KBLI.JK', 'KIJA.JK', 'KKGI.JK', 'KLBF.JK', 'KPIG.JK', 'LPKR.JK', 'LPPF.JK', 'LSIP.JK', 'LTLS.JK', 'MAIN.JK', 'MAPI.JK', 'MAYA.JK', 'MBSS.JK', 'MCOR.JK', 'MDKA.JK', 'MEDC.JK', 'MEGA.JK', 'MIDI.JK', 'MIKA.JK', 'MLBI.JK', 'MLIA.JK', 'MLPL.JK', 'MMLP.JK', 'MNCN.JK', 'MPMX.JK', 'MREI.JK', 'MTDL.JK', 'MTLA.JK', 'MYOR.JK', 'NISP.JK', 'PANR.JK', 'PANS.JK', 'PGAS.JK', 'PNBN.JK', 'PNIN.JK', 'PNLF.JK', 'PTBA.JK', 'PTPP.JK', 'PTRO.JK', 'PWON.JK', 'RAJA.JK', 'RALS.JK', 'SAME.JK', 'SCMA.JK', 'SGRO.JK', 'SIDO.JK', 'SILO.JK', 'SIMP.JK', 'SMAR.JK', 'SMBR.JK', 'SMDR.JK', 'SMGR.JK', 'SMMA.JK', 'SMRA.JK', 'SMSM.JK', 'SRTG.JK', 'SSIA.JK', 'SSMS.JK', 'TBIG.JK', 'TBLA.JK', 'TINS.JK', 'TKIM.JK', 'TLKM.JK', 'TMAS.JK', 'TOBA.JK', 'TOTL.JK', 'TOWR.JK', 'TPMA.JK', 'TRIM.JK', 'TSPC.JK', 'ULTJ.JK', 'UNIC.JK', 'UNTR.JK', 'UNVR.JK', 'VICO.JK', 'WIIM.JK', 'WINS.JK', 'WTON.JK', 'SHIP.JK', 'POWR.JK', 'PRDA.JK', 'BRIS.JK', 'CARS.JK', 'CLEO.JK', 'WOOD.JK', 'HRTA.JK', 'MARK.JK', 'MCAS.JK', 'PSSI.JK', 'MORA.JK', 'PBID.JK', 'IPCM.JK', 'BTPS.JK', 'SPTO.JK', 'HEAL.JK', 'TUGU.JK', 'MSIN.JK', 'MAPA.JK', 'IPCC.JK', 'FILM.JK', 'PANI.JK', 'GOOD.JK', 'SKRN.JK', 'BOLA.JK', 'KOTA.JK', 'KEEN.JK', 'TEBE.JK', 'KEJU.JK', 'PSGO.JK', 'UCID.JK', 'CSRA.JK', 'SAMF.JK', 'SGER.JK', 'PNGO.JK', 'BBSI.JK', 'VICI.JK', 'WMUU.JK', 'UNIQ.JK', 'TAPG.JK', 'BMHS.JK', 'MCOL.JK', 'GTSI.JK', 'MTEL.JK', 'CMRY.JK', 'RMKE.JK', 'AVIA.JK', 'DRMA.JK', 'ADMR.JK', 'STAA.JK', 'MTMH.JK', 'TRGU.JK', 'HATM.JK', 'JARR.JK', 'ELPI.JK', 'MKTR.JK', 'OMED.JK', 'SUNI.JK', 'PGEO.JK', 'BDKR.JK', 'CUAN.JK', 'SMIL.JK', 'AMMN.JK', 'MAHA.JK', 'ERAL.JK', 'BREN.JK', 'MSTI.JK', 'ALII.JK', 'GOLF.JK', 'DAAZ.JK', 'AADI.JK', 'MDIY.JK', 'DGWG.JK', 'CBDK.JK', 'MINE.JK', 'PSAT.JK', 'BLOG.JK', 'YUPI.JK', 'MDLA.JK', 'NCKL.JK', 'MBMA.JK', 'RAAM.JK', 'ADRO.JK', 'AGRO.JK']
    return fallback


In [ ]:
# [BRAIN] SECTION 3: CORE LOGIC (Institutional Predictors & Meta-Label Gate)
import ta

def frac_diff(series, d=0.4, window=100):
    w = [1.0]
    for k in range(1, window):
        w.append(-w[-1] * (d - k + 1) / k)
    w = np.array(w[::-1])
    result = np.full(len(series), np.nan)
    for t in range(window - 1, len(series)):
        result[t] = np.dot(w, series[t - window + 1:t + 1])
    return result

class MLPredictor:
    def __init__(self, path):
        brain = joblib.load(path)
        self.model = brain.get('model')
        self.features = brain.get('features', [])
        print(f'[OK] LightGBM Model loaded | {len(self.features)} features')

    def build_features(self, df):
        close = df['Close'].squeeze()
        high = df['High'].squeeze()
        low = df['Low'].squeeze()
        volume = df['Volume'].squeeze()
        feat = pd.DataFrame(index=df.index)
        feat['Returns'] = close.pct_change()
        feat['RSI'] = ta.momentum.RSIIndicator(close=close, window=14).rsi()
        macd = ta.trend.MACD(close=close)
        feat['MACD'] = macd.macd_diff()
        boll = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
        feat['BB_High'] = boll.bollinger_hband_indicator()
        feat['BB_Low'] = boll.bollinger_lband_indicator()
        feat['ATR'] = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
        feat['ADX'] = ta.trend.ADXIndicator(high=high, low=low, close=close, window=14).adx()
        obv = ta.volume.OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()
        feat['OBV_norm'] = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-7)
        feat['day_of_week'] = df.index.dayofweek
        feat['week_of_month'] = (df.index.day - 1) // 7
        feat['frac_diff_close'] = frac_diff(close.values.flatten(), d=0.4, window=100)
        vol_ma = volume.rolling(20).mean()
        feat['vol_ratio'] = volume / (vol_ma + 1e-7)
        return feat.dropna()

    def predict(self, df):
        try:
            feat = self.build_features(df)
            if len(feat) == 0: return 0, 0.0
            row = feat[self.features].tail(1)
            pred = self.model.predict(row)[0]
            proba = self.model.predict_proba(row)[0]
            return int(pred), float(proba[1])
        except: return 0, 0.0

class CNNPredictor:
    def __init__(self, path):
        self.interpreter = tflite.Interpreter(model_path=path)
        self.interpreter.allocate_tensors()
        print('[OK] CNN TFLite loaded (5-channel OHLCV)')

    def predict(self, df):
        try:
            close = df['Close'].squeeze().values[-30:]
            high = df['High'].squeeze().values[-30:]
            low = df['Low'].squeeze().values[-30:]
            volume = df['Volume'].squeeze().values[-30:]
            opn = df['Open'].squeeze().values[-30:]
            raw = np.column_stack([opn, high, low, close, volume])
            seq_min, seq_max = raw.min(axis=0), raw.max(axis=0)
            norm_seq = (raw - seq_min) / (seq_max - seq_min + 1e-7)
            
            input_details = self.interpreter.get_input_details()
            input_data = np.expand_dims(norm_seq.astype(np.float32), axis=0)
            self.interpreter.set_tensor(input_details[0]['index'], input_data)
            self.interpreter.invoke()
            output = self.interpreter.get_tensor(self.interpreter.get_output_details()[0]['index'])[0]
            return float(output[1]) if len(output) > 1 else float(output[0])
        except Exception as e:
            print(f'[WARN] CNN predict error: {e}')
            return 0.5


In [ ]:
# [RUN] SECTION 4: MAIN EXECUTION (Recursive Discovery & Polling)
def find_model_file(filename):
    search_paths = ['/kaggle/input', '/kaggle/working', '.']
    for root_dir in search_paths:
        if not os.path.exists(root_dir): continue
        for root, dirs, files in os.walk(root_dir):
            if filename in files: return os.path.join(root, filename)
    return None

def get_market_regime():
    try:
        idx = yf.download('^JKSE', period='250d', progress=False, auto_adjust=True)
        close = idx['Close'].squeeze()
        sma200 = close.tail(200).mean()
        curr = close.iloc[-1]
        return 'BULL' if curr > sma200 else 'BEAR'
    except: return 'UNKNOWN'

def run_inference():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    
    lgbm_path = find_model_file('glu_brain_v1.joblib')
    cnn_path = find_model_file('cnn_daily_t2.tflite')
    
    if not lgbm_path or not cnn_path:
        print(f'[ERROR] Models missing! Ensure 00a and 00b outputs are attached.')
        return
    
    queue = fb.wait_for_queue('research')
    if not queue: 
        print('[EMPTY] Research queue empty after timeout.')
        return
        
    candidates = []
    for q in queue: 
        if isinstance(q, list): candidates.extend(q)
        else: candidates.append(q)
    
    regime = get_market_regime()
    gate = 0.60 if regime == 'BULL' else 0.70
    print(f'[MARKET] Regime: {regime} | Meta-Gate: {gate:.0%}')
    
    lgbm = MLPredictor(lgbm_path)
    cnn = CNNPredictor(cnn_path)
    signals = {}
    
    target_tickers = list(set(candidates))
    total = len(target_tickers)
    
    for i, ticker in enumerate(target_tickers):
        try:
            print(f'[INFERENCE] ({i+1}/{total}) Analyzing {ticker}...', flush=True)
            df = yf.download(ticker, period='150d', progress=False, auto_adjust=True)
            if len(df) < 120: continue
            
            lgbm_buy, lgbm_prob = lgbm.predict(df)
            cnn_prob = cnn.predict(df)
            
            if lgbm_buy == 1 and cnn_prob >= gate:
                signals[ticker] = {
                    'price': float(df['Close'].iloc[-1]),
                    'lgbm_confidence': float(lgbm_prob),
                    'cnn_confidence': float(cnn_prob),
                    'regime': regime,
                    'timestamp': datetime.now().isoformat()
                }
                print(f'[SIGNAL] {ticker} APPROVED ({lgbm_prob:.0%} | {cnn_prob:.0%})')
        except Exception as e: print(f'[WARN] Error {ticker}: {e}')
            
    if signals:
        fb.push_task('signals', signals)
        fb.log_event('INFERENCE', f'Signals Found: {len(signals)}')
        print(f'[OK] Dispatched {len(signals)} signals.')
    else:
        print('[BLOCK] No high-conviction signals found.')

run_inference()